# silver_curated_weather

Reads the daily weather Delta table from `silver_raw` (shortcut into the bronze lakehouse `dbo.weather`),
cleans it, and writes `weather_daily` into `silver_curated` with conformed data-science friendly columns.

Outputs (in `silver_curated/Tables/dbo/weather_daily`):
- date, store_id, city, state, latitude, longitude
- temperature_max_c/f, temperature_min_c/f, temperature_avg_c/f
- precipitation_mm, snowfall_cm, wind_speed_max_kmh
- weather_code, weather_description
- Derived flags: is_rainy (>= 1mm precip), is_snowy (> 0 snow), is_hot (max_c >= 30), is_cold (min_c <= 0)

Source is already daily-grain, so this is a clean+enrich pass. Re-runnable via overwrite.

In [ ]:
# Parameters baked by deploy.ps1.
silver_raw_workspace_id     = ""
silver_raw_lakehouse_id     = ""
silver_curated_workspace_id = ""
silver_curated_lakehouse_id = ""

In [ ]:
from pyspark.sql import functions as F

for n,v in [('silver_raw_workspace_id',silver_raw_workspace_id),('silver_raw_lakehouse_id',silver_raw_lakehouse_id),('silver_curated_workspace_id',silver_curated_workspace_id),('silver_curated_lakehouse_id',silver_curated_lakehouse_id)]:
    if not v: raise ValueError(f'{n} parameter is required')

src_path = f'abfss://{silver_raw_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_raw_lakehouse_id}/Tables/dbo/weather'
tgt_path = f'abfss://{silver_curated_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_curated_lakehouse_id}/Tables/dbo/weather_daily'

w = spark.read.format('delta').load(src_path)

daily = (w
    .withColumn('temperature_avg_c', (F.col('temperature_max_c') + F.col('temperature_min_c')) / F.lit(2.0))
    .withColumn('temperature_avg_f', (F.col('temperature_max_f') + F.col('temperature_min_f')) / F.lit(2.0))
    .withColumn('is_rainy', F.coalesce(F.col('precipitation_mm'),    F.lit(0.0)) >= F.lit(1.0))
    .withColumn('is_snowy', F.coalesce(F.col('snowfall_cm'),         F.lit(0.0)) >  F.lit(0.0))
    .withColumn('is_hot',   F.coalesce(F.col('temperature_max_c'),   F.lit(-999.0)) >= F.lit(30.0))
    .withColumn('is_cold',  F.coalesce(F.col('temperature_min_c'),   F.lit(999.0))  <= F.lit(0.0))
)

(daily.write.format('delta')
    .mode('overwrite').option('overwriteSchema','true')
    .save(tgt_path))
print(f'wrote {daily.count():,} rows -> {tgt_path}')

In [ ]:
out = spark.read.format('delta').load(tgt_path)
print(f'total rows: {out.count():,}')
out.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date'), F.countDistinct('store_id').alias('stores')).show(truncate=False)
out.orderBy(F.desc('date'), 'store_id').show(5, truncate=False)